In [33]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import json
import pandas as pd

In [34]:
df_business_raw = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "yelp-dataset/yelp-dataset",
    "yelp_academic_dataset_business.json",
    pandas_kwargs={"lines": True},
)

# only care about Santa Barbara for now
df_business = df_business_raw[df_business_raw["city"]== "Santa Barbara"].copy()


In [35]:
# the categories that we want to include

allowed_categories = set(
    pd.read_csv("../processed/categories.csv")["category"]
    .dropna()
    .str.strip()
)

len(allowed_categories)

33

In [36]:
category_boolean = df_business["categories"].fillna("").apply(
    lambda value: any(
        category.strip() in allowed_categories
        for category in value.split(",")
    )
)

category_boolean

0         False
26        False
85         True
91        False
120       False
          ...  
150239    False
150264     True
150284    False
150290    False
150303    False
Name: categories, Length: 3829, dtype: bool

In [37]:
df_business = df_business.loc[category_boolean].copy()

In [38]:
df_business.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
85,IDtLPgUrqorrpqSLdfMhZQ,Helena Avenue Bakery,"131 Anacapa St, Ste C",Santa Barbara,CA,93101,34.414445,-119.690672,4.0,389,1,"{'RestaurantsTakeOut': 'True', 'NoiseLevel': '...","Food, Restaurants, Salad, Coffee & Tea, Breakf...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-14:0', '..."
141,SZU9c8V2GuREDN5KgyHFJw,Santa Barbara Shellfish Company,230 Stearns Wharf,Santa Barbara,CA,93101,34.408715,-119.685019,4.0,2404,1,"{'OutdoorSeating': 'True', 'RestaurantsAttire'...","Live/Raw Food, Restaurants, Seafood, Beer Bar,...","{'Monday': '0:0-0:0', 'Tuesday': '11:0-21:0', ..."
691,-kY_HDP7IMvGl-kBIZVU4A,Dune Coffee Roasters - Anacapa,528 Anacapa St,Santa Barbara,CA,93101,34.418994,-119.695068,4.0,320,1,"{'RestaurantsPriceRange2': '2', 'BusinessAccep...","Coffee & Tea, Coffee Roasteries, Food","{'Monday': '6:0-17:0', 'Tuesday': '6:0-17:0', ..."
946,-ujBP1Dw0j1-Ffaz97-LXQ,Lama Dog Tap Room,116 Santa Barbara St,Santa Barbara,CA,93101,34.415747,-119.688480,4.0,238,1,"{'RestaurantsReservations': 'False', 'WiFi': '...","Beer, Wine & Spirits, Pubs, Arts & Entertainme...","{'Monday': '0:0-0:0', 'Tuesday': '11:30-17:0',..."
1320,UFpCraqzFBAhtZqmxmiWsA,Cat Therapy,"1213 State St, Ste L",Santa Barbara,CA,93101,34.423302,-119.705471,4.5,116,1,"{'WheelchairAccessible': 'True', 'WiFi': 'u'fr...","Themed Cafes, Cafes, Pets, Arts & Entertainmen...","{'Monday': '10:0-17:15', 'Wednesday': '10:0-19..."


In [39]:
from pathlib import Path
from pprint import pformat

business = df_business.loc[
    df_business["name"].str.strip().eq("Cat Therapy")
].iloc[0]

output_path = Path("business_example_Cat_Therapy.txt")
output_path.write_text(
    pformat(business["attributes"], sort_dicts=False),
    encoding="utf-8",
)

431

In [40]:
attribute_keys = (
    pd.read_csv("attributes.csv")["attribute"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Handle missing dictionaries without changing the original column.
safe_attributes = df_business["attributes"].apply(
    lambda value: value if isinstance(value, dict) else {}
)

attributes_df = pd.DataFrame(
    safe_attributes.tolist(),
    index=df_business.index,
).reindex(columns=attribute_keys)

for key in attribute_keys:
    df_business[f"attribute_{key}"] = attributes_df[key]

In [41]:
df_business.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,attribute_RestaurantsCounterService,attribute_RestaurantsDelivery,attribute_RestaurantsGoodForGroups,attribute_RestaurantsPriceRange2,attribute_RestaurantsReservations,attribute_RestaurantsTableService,attribute_RestaurantsTakeOut,attribute_Smoking,attribute_WheelchairAccessible,attribute_WiFi
85,IDtLPgUrqorrpqSLdfMhZQ,Helena Avenue Bakery,"131 Anacapa St, Ste C",Santa Barbara,CA,93101,34.414445,-119.690672,4.0,389,...,NaN,None,True,2,False,False,True,NaN,True,u'no'
141,SZU9c8V2GuREDN5KgyHFJw,Santa Barbara Shellfish Company,230 Stearns Wharf,Santa Barbara,CA,93101,34.408715,-119.685019,4.0,2404,...,NaN,True,False,2,False,NaN,True,NaN,NaN,u'no'
691,-kY_HDP7IMvGl-kBIZVU4A,Dune Coffee Roasters - Anacapa,528 Anacapa St,Santa Barbara,CA,93101,34.418994,-119.695068,4.0,320,...,NaN,NaN,NaN,2,NaN,NaN,True,NaN,NaN,u'free'
946,-ujBP1Dw0j1-Ffaz97-LXQ,Lama Dog Tap Room,116 Santa Barbara St,Santa Barbara,CA,93101,34.415747,-119.688480,4.0,238,...,NaN,True,True,2,False,NaN,True,u'no',True,'free'
1320,UFpCraqzFBAhtZqmxmiWsA,Cat Therapy,"1213 State St, Ste L",Santa Barbara,CA,93101,34.423302,-119.705471,4.5,116,...,NaN,NaN,NaN,1,NaN,NaN,False,NaN,True,u'free'


In [42]:
category_names = (
    pd.read_csv("../processed/categories.csv")["category"])

allowed_categories = set(category_names)

# Split each business's categories and remove surrounding spaces.
business_categories = df_business["categories"].fillna("").apply(
    lambda value: {
        category.strip()
        for category in value.split(",")
        if category.strip()
    }
)

# True when the business has that exact category.
for category in category_names:
    df_business[f"category_{category}"] = business_categories.apply(
        lambda values: category in values
    )

# True when at least one category is outside the CSV list.
df_business["category_other"] = business_categories.apply(
    lambda values: bool(values - allowed_categories)
)

In [43]:
df_business.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,category_Bookstores,category_Used Bookstore,category_Community Centers,category_Hotels,category_Hotel bar,category_Lounges,category_Wine Bars,category_Pubs,category_Gastropubs,category_other
85,IDtLPgUrqorrpqSLdfMhZQ,Helena Avenue Bakery,"131 Anacapa St, Ste C",Santa Barbara,CA,93101,34.414445,-119.690672,4.0,389,...,False,False,False,False,False,False,False,False,False,True
141,SZU9c8V2GuREDN5KgyHFJw,Santa Barbara Shellfish Company,230 Stearns Wharf,Santa Barbara,CA,93101,34.408715,-119.685019,4.0,2404,...,False,False,False,False,False,False,False,False,False,True
691,-kY_HDP7IMvGl-kBIZVU4A,Dune Coffee Roasters - Anacapa,528 Anacapa St,Santa Barbara,CA,93101,34.418994,-119.695068,4.0,320,...,False,False,False,False,False,False,False,False,False,True
946,-ujBP1Dw0j1-Ffaz97-LXQ,Lama Dog Tap Room,116 Santa Barbara St,Santa Barbara,CA,93101,34.415747,-119.688480,4.0,238,...,False,False,False,False,False,False,False,True,False,True
1320,UFpCraqzFBAhtZqmxmiWsA,Cat Therapy,"1213 State St, Ste L",Santa Barbara,CA,93101,34.423302,-119.705471,4.5,116,...,False,False,False,False,False,False,False,False,False,True


In [44]:
days = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday",
]

# Handle missing hours without changing the original column.
safe_hours = df_business["hours"].apply(
    lambda value: value if isinstance(value, dict) else {}
)

for day in days:
    day_hours = safe_hours.apply(lambda hours: hours.get(day))

    split_hours = (
        day_hours.astype("string")
        .str.split("-", n=1, expand=True)
        .reindex(columns=[0, 1])
    )

    df_business[f"{day}_start"] = split_hours[0]
    df_business[f"{day}_end"] = split_hours[1]

In [45]:
df_business.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,Wednesday_start,Wednesday_end,Thursday_start,Thursday_end,Friday_start,Friday_end,Saturday_start,Saturday_end,Sunday_start,Sunday_end
85,IDtLPgUrqorrpqSLdfMhZQ,Helena Avenue Bakery,"131 Anacapa St, Ste C",Santa Barbara,CA,93101,34.414445,-119.690672,4.0,389,...,8:0,14:0,8:0,14:0,8:0,14:0,8:0,14:0,8:0,14:0
141,SZU9c8V2GuREDN5KgyHFJw,Santa Barbara Shellfish Company,230 Stearns Wharf,Santa Barbara,CA,93101,34.408715,-119.685019,4.0,2404,...,11:0,21:0,11:0,14:0,11:0,21:0,11:0,21:0,11:0,21:0
691,-kY_HDP7IMvGl-kBIZVU4A,Dune Coffee Roasters - Anacapa,528 Anacapa St,Santa Barbara,CA,93101,34.418994,-119.695068,4.0,320,...,6:0,17:0,6:0,17:0,6:0,17:0,6:0,17:0,7:0,17:0
946,-ujBP1Dw0j1-Ffaz97-LXQ,Lama Dog Tap Room,116 Santa Barbara St,Santa Barbara,CA,93101,34.415747,-119.688480,4.0,238,...,11:30,22:0,11:30,20:0,11:30,0:0,11:30,0:0,11:30,22:0
1320,UFpCraqzFBAhtZqmxmiWsA,Cat Therapy,"1213 State St, Ste L",Santa Barbara,CA,93101,34.423302,-119.705471,4.5,116,...,10:0,19:0,10:0,19:0,10:0,20:0,10:0,20:0,10:0,19:0


In [ ]:
#df_reviews = kagglehub.dataset_load(
#    KaggleDatasetAdapter.PANDAS,
#    "yelp-dataset/yelp-dataset",
#    "yelp_academic_dataset_review.json")


In [48]:
business_ids = set(df_business["business_id"])

# Download the source file or reuse its cached copy.
reviews_path = kagglehub.dataset_download(
    "yelp-dataset/yelp-dataset",
    path="yelp_academic_dataset_review.json",
)

matching_chunks = []

# Read 10,000 reviews at a time.
with pd.read_json(
    reviews_path,
    lines=True,
    chunksize=10_000,
) as reader:
    for chunk in reader:
        selected = chunk.loc[
            chunk["business_id"].isin(business_ids)
        ]

        if not selected.empty:
            matching_chunks.append(selected.copy())

# Combine only the matching reviews.
df_reviews = (
    pd.concat(matching_chunks, ignore_index=True)
    if matching_chunks
    else pd.DataFrame()
)

del matching_chunks

print(f"Loaded {len(df_reviews):,} matching reviews.")
df_reviews.head(10)

Loaded 110,027 matching reviews.


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,L0jv8c2FbpWSlfNC6bbUEA,bFPdtzu11Oi0f92EAcjqmg,IDtLPgUrqorrpqSLdfMhZQ,5,0,0,0,What a great addition to the Funk Zone! Grab ...,2016-10-13 22:50:47
1,4zopEEPqfwm-c_FNpeHZYw,JYYYKt6TdVA4ng9lLcXt_g,SZU9c8V2GuREDN5KgyHFJw,5,0,0,0,We were a bit weary about trying the Shellfish...,2016-05-31 02:14:54
2,SpmKsFnwg35PTgCVujOOJA,RiQWbgwR-asiU_mA7s8gpg,pUVRCYL8rT4I5Ry5FYkRsA,5,2,0,1,Sweet spot in the Waterline. Came for some liv...,2016-12-19 22:58:19
3,zM7u3iffMjniyvQAaN9Tnw,QacYrFyCrFLmC726YEEBsA,29YqJwOGEuAWqlHZxMc1OA,1,0,1,0,So disappointing on so many levels. Have been ...,2012-05-29 03:51:08
4,Gczff1Qxg4WVmlBLPrZwzQ,cpp89UW0imv45y3a_O444w,34Eqv8jXgxg_EEwcsNgeeg,5,0,0,0,We absolutely love everything we have tried he...,2014-05-25 22:33:02
5,bi6GaeWDGceGv62lXTIKQA,RgtbLaiU22zqaCk20HgbiQ,bjhCtlYHrkgA5Ku8l-rB3g,1,1,0,0,Very disappointed. We went to eat at 2:15 on ...,2015-01-04 02:26:46
6,H4mSxGeKTv4VFcc9YUmtpQ,oGV84fsuRbLQRRQ6ZZC1dA,-kY_HDP7IMvGl-kBIZVU4A,4,0,0,0,The espresso here is excellent. My favorite pl...,2016-07-14 19:43:42
7,YS8WO_OW6oYFQhH85aRXew,I7ugU8kU4bPh4UL90yJanQ,eL4lyE7LNoXEMvpcJ8WNVw,5,0,0,0,"Great, beautiful spot to do wine tasting. I lo...",2016-12-04 16:55:51
8,cC79tWLtH1U1n-oi7EozMw,FNf6kM1wfeBeB2UW6pif7g,qdpdUjdkEUpzYDdce2yZng,4,0,0,0,"The line was out the door, but I had a sandwic...",2016-11-09 19:58:45
9,dpgq69tsI2f_GpRH71hQsA,Uk3X2AypU8AqvcYEVf7s6Q,eL4lyE7LNoXEMvpcJ8WNVw,3,5,0,0,This was a tough one! On the merits of the w...,2011-08-25 16:14:24


In [47]:
df_reviews.shape 

NameError: name 'df_reviews' is not defined